# 🔬 ANN Architecture Explorer — Lite
### Quick Neural Network Configuration Comparison

---

**Purpose:** Quickly explore different ANN architectures and hyperparameters with interactive comparison plots. This is the lightweight version — no file exports, no config validation, just train and compare.

For persistent results with CSV/JSON export, multi-run tracking, and HTML reports, use the full **ANN Architecture Explorer v4**.

**How to use this notebook:**
1. **Section 1** — Load your dataset (two samples provided)
2. **Section 2** — Define your configurations
3. **Section 3+** — Run all cells below without changes

**Data Split Strategy:**
```
Full Dataset
├── Train  (e.g. 70%)  → gradient updates during training
├── Dev    (e.g. 15%)  → per-epoch accuracy monitoring & config selection
└── Test   (e.g. 15%)  → touched ONCE at the end for unbiased final evaluation
```

Alternatively, enable **k-fold cross-validation** for small datasets.

<br>

#### **IMPORTANT:** <br>No code changes needed below Section 2.

---
## Section 0 — Import (Do not edit)

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split, KFold
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time
import scipy.stats as stats
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    import matplotlib_inline.backend_inline
    matplotlib_inline.backend_inline.set_matplotlib_formats('svg')
except:
    pass


print('All imports successful.')

All imports successful.


---
## Section 1 — Load Data (Edit optional in Section 1b)

The following datasets are pre-loaded for testing:

- The **Iris** dataset (150 flowers, 4 botanical features, 3 species -> multiclass classification).
- The **Wine Quality** dataset (1,599 red wines, 11 chemical features, binary quality label -> binary classification).

To use your own data, replace `features` and `labels`:
- `features`: shape `(n_samples, n_features)`, dtype `float32`
- `labels`: shape `(n_samples, 1)` for binary, shape `(n_samples,)` dtype `long` for multiclass

### Section 1a - Sample Datasets (Do not edit)

In [2]:
# (Do not edit this cell)
# ===============================================================
# SAMPLE DATASET — 1: Iris (3-class classification)
# ===============================================================

# --- load sample dataset (iris dataset pre-loaded for testing) ---
url = 'https://raw.githubusercontent.com/mwaskom/seaborn-data/master/iris.csv'
name = url.split('/')[-1].split('.')[0].capitalize()
data = pd.read_csv(url)

# --- features: Z-score normalize and extract botanical features ---
cols2zscore = data.keys().drop('species')
data[cols2zscore] = data[cols2zscore].apply(stats.zscore)
features = data[cols2zscore].values

# --- labels: Create integer labels ---
data['intSpecies'] = data['species'].astype('category').cat.codes
labels = data['intSpecies'].values

# --- create labels dict for clear mapping ---
label_dict = dict(enumerate(data['species'].astype('category').cat.categories))
# or manually:
# label_dict = {
#     0: "setosa",
#     1: "versicolor",
#     2: "virginica"
# }

# --- print dataset summary for verification --- 
print(f'Loaded {name} dataset with {features.shape[0]} samples, {features.shape[1]} features, and {len(label_dict)} classes.')
print(f'Labels (species): {np.bincount(labels)} (0=setosa, 1=versicolor, 2=virginica)')

Loaded Iris dataset with 150 samples, 4 features, and 3 classes.
Labels (species): [50 50 50] (0=setosa, 1=versicolor, 2=virginica)


In [3]:
# (Do not edit this cell)
# ============================================================
# SAMPLE DATASET — 2: Wine Quality (Binary classification)
# ============================================================

# --- load sample dataset (wine quality dataset pre-loaded for testing) ---
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/wine-quality/winequality-red.csv'
name = url.split('/')[-1].split('.')[0].capitalize()
data = pd.read_csv(url, sep=';')

# --- clean data ---
data = data[data['total sulfur dioxide'] < 200] # filter out outliers for better training

# --- features: Z-score normalize and extract chemical features ---
cols2zscore = data.keys().drop('quality')
data[cols2zscore] = data[cols2zscore].apply(stats.zscore)
features = data[cols2zscore].values

# --- labels: Create binary labels: 1 for quality > 5, else 0 ---
data['boolQuality'] = (data['quality'] > 5).astype(int)
labels = data['boolQuality'].values

# --- create labels dict for clear mapping ---
label_dict = {
    0: "bad",
    1: "good"
}

# --- print dataset summary for verification --- 
print(f'Loaded {name} dataset with {features.shape[0]} samples, {features.shape[1]} features, and {len(label_dict)} classes.')
print(f'Labels (quality): {np.bincount(labels)} (0=bad, 1=good)')

Loaded Winequality-red dataset with 1597 samples, 11 features, and 2 classes.
Labels (quality): [744 853] (0=bad, 1=good)


In [4]:
# (Do not edit this cell)
# ============================================================
# SAMPLE DATASET — 3: Auto MPG (Regression)
# ============================================================

# --- load sample dataset (auto mpg dataset pre-loaded for testing) ---
url = 'https://archive.ics.uci.edu/ml/machine-learning-databases/auto-mpg/auto-mpg.data'
name = url.split('/')[-1].split('.')[0].capitalize()
col_names = ['mpg','cylinders','displacement','horsepower','weight','acceleration','model_year','origin','car_name']

# --- clean data ---
data = pd.read_csv(url, sep=r'\s+', names=col_names, na_values='?')
data = data.dropna() # drop rows with missing values
data = data.drop(columns=['car_name'])

# --- features: Z-score normalize all predictor columns ---
cols2zscore = data.keys().drop('mpg')
data[cols2zscore] = data[cols2zscore].apply(stats.zscore)
features = data[cols2zscore].values

# --- labels: continuous miles-per-gallon ---
labels = data['mpg'].values
label_dict = None # no label dict needed for regression

# --- print dataset summary for verification ---
print(f'Loaded {name} dataset with {features.shape[0]} samples and {features.shape[1]} features.')
print(f'Labels (mpg): min={labels.min():.1f}, max={labels.max():.1f}, mean={labels.mean():.1f}, std={labels.std():.1f}, total unique values={len(np.unique(labels))}')

Loaded Auto-mpg dataset with 392 samples and 7 features.
Labels (mpg): min=9.0, max=46.6, mean=23.4, std=7.8, total unique values=127


### Section 1b - Load Your Own Dataset (EDIT OPTIONAL)

In [ ]:
# (Feel free to edit this cell if you want to load your own dataset)
# =================================
# LOAD YOUR OWN DATASET 
# =================================
# Load your own dataset here by providing `features` and `labels` as arrays. You can use pandas to load and preprocess your data, then extract the features and labels as NumPy arrays. 
# Make sure to follow the format of the sample datasets above for consistency. For conveniece a template is provided below. 
# You can also copy-paste the relevant code from the sample dataset cells above to help structure your data loading and preprocessing steps.
# IMPORTANT: Make sure to extract features and labels as arrays as they will be converted to PyTorch tensors later in the script.


# --- Load your data ---
# data =                                # type: array-like or pandas DataFrame
# name = 'Your Dataset Name'            # type: string

# --- Extract features ---
# features =                            # type: array-like or pandas DataFrame  

# --- Extract or create labels ---
# labels =                             # type: array-like or pandas Series


# --- print final dataset summary for verification --- 
print(f'Loaded {name} dataset with {features.shape[0]} samples, {features.shape[1]} features and {len(np.unique(labels))} unique labels.')

Loaded Auto-mpg dataset with 392 samples, 7 features and 127 unique labels.


### Section 1c — Verify dataset (Do not edit)

In [6]:
# (Do not edit this cell)
# ==================================================================== 
# Verify dataset, extract information, and convert to PyTorch tensors
# ====================================================================

# --- Convert to PyTorch tensors ---
T_features = torch.tensor(features).float()
T_labels   = torch.tensor(labels).float()[:, None]

# --- Extract dataset metadata ---
n_samples = T_features.shape[0]
n_features = T_features.shape[1]
n_classes = len(torch.unique(T_labels))
dist_classes = torch.bincount(T_labels.squeeze().long())

# --- Determine loss type ---
reg_threshold = 50  # heuristic threshold for regression vs classification (can be adjusted based on dataset size and expected number of classes)
if n_classes > reg_threshold:
    print(f"⚠️ {n_classes} unique targets found — likely regression. Progressing with 1 output and regression loss.\n")
    n_classes = 1
    losstype = "regression"
elif n_classes == 2:
    print(f"{n_classes} classes found. Treating as binary classification. Progressing with 1 output.\n")
    n_classes = 1
    losstype = "binary"
elif n_classes >= 3:
    print(f"{n_classes} classes found. Treating as multi-class classification. Progressing with {n_classes} outputs.\n")
    losstype = "multiclass"
    T_labels = torch.tensor(labels).long()  # convert labels to long for multiclass classification (needed for CrossEntropyLoss)
else:
    print(f"⚠️ Only {n_classes} class found — check your labels.\n")
    losstype = "⚠️ uniform class distribution - no suitable loss function"  # fallback


# --- Collect dataset information ---
rows2print = [
    ("Dataset Name", f"{name}"),
    ("Dataset Info", f"({n_samples} samples, {n_features} features, {n_classes} classes)"),
    ("Loss type", losstype.capitalize() if losstype in ['regression', 'binary', 'multiclass'] else losstype),
    ("Features tensor shape", f"{tuple(T_features.shape)} (samples, features)"),
    ("Labels tensor shape", f"{tuple(T_labels.shape)} (samples, 1)"),
]
# --- Add class distribution to dataset information for classification ---
if losstype == "binary" or losstype == "multiclass":
    rows2print.append(("Class distribution", f"{dist_classes.tolist()}"))

# --- Add label mapping to dataset information ---
if "label_dict" in globals() and isinstance(label_dict, dict):
    spacer = max(len(v) for v in label_dict.values()) + 2 # calculate spacer based on longest labelname
    for i, (k, v) in enumerate(sorted(label_dict.items())):
        header = "Label mapping" if i == 0 else ""
        samplecount = (T_labels.squeeze().long() == k).sum().item() # counted number of samples for this label (used instead of iterating over dist_classes in case of non-sequential labels)
        rows2print.append((header, f"{k:<2} -> {v:<{spacer}} | {samplecount} samples"))

# --- Print dataset information ---
spacer = max(len(r[0]) for r in rows2print) + 2 # calculate spacer based on longest header
for header, value in rows2print:
    print(f"{header:<{spacer}} {value}")

⚠️ 127 unique targets found — likely regression. Progressing with 1 output and regression loss.

Dataset Name            Auto-mpg
Dataset Info            (392 samples, 7 features, 1 classes)
Loss type               Regression
Features tensor shape   (392, 7) (samples, features)
Labels tensor shape     (392, 1) (samples, 1)


---
## Section 2 — Define Your Configurations (Edit optional in Section 2b)

**HOW TO USE: You can modify any of the configurations below to experiment with different architectures and hyperparameters or test the notebook first with the 5 provided examples.**

### Parameter Reference

| Parameter | What it does | Options | Gold Standard Default |
|---|---|---|---|
| `layer_sizes` | Network architecture | Any list of ints, e.g. `[11,64,64,1]` | `[n_features, 32, 32, 1]` |
| `actfun` | Activation function | `'relu'`, `'tanh'`, `'sigmoid'`, `'leaky_relu'`, `'gelu'`, `'elu'`, `'selu'`, `'silu'` | `'relu'` |
| `leaky_slope` | LeakyReLU negative slope | 0.001 – 0.3 | `0.01` |
| `learning_rate` | Optimizer step size | 0.00001 – 0.1 | `0.001` (Adam) |
| `num_epochs` | Training passes | 100 – 5000 | `1000` |
| `batch_size` | Samples per update | 8, 16, 32, 64, 128 | `32` |
| `optimizer` | Update algorithm | `'adam'`, `'adamw'`, `'sgd'`, `'rmsprop'`, `'adagrad'` | `'adam'` |
| `momentum` | SGD/RMSprop momentum | 0.0 – 0.99 | `0.9` |
| `loss_function` | Error measurement | Binary: `'bce_logits'` · Multiclass: `'cross_entropy'` · Regression: `'mse'`, `'l1'`, `'huber'` | `'bce_logits'` |
| `weight_decay` | L2 regularization | 0.0 – 0.01 | `1e-4` |
| `l1_lambda` | L1 regularization | 0.0 – 0.001 | `0.0` |
| `dropout_rate` | Neuron drop fraction | 0.0 – 0.5 | `0.2` |
| `use_batchnorm` | Batch normalization | `True`, `False` | `True` |

### Data Split Parameters

| Parameter | What it does | Gold Standard |
|---|---|---|
| `dev_size` | Fraction for dev/validation set (per-epoch monitoring & config selection) | `0.15` |
| `test_size` | Fraction for final held-out test set (touched ONCE at the end) | `0.15` |
| `use_kfold` | Use k-fold CV instead of single split (better for small datasets <1000) | `False` |
| `n_folds` | Number of folds for k-fold CV (only used if `use_kfold=True`) | `5` |

### Section 2a - Sample Configurations (Do not edit)

In [7]:
# (Do not edit this cell)
# ============================================================
# SAMPLE CONFIGURATIONS
# ============================================================

# **Note on Config D:** 
# This configuration intentionally uses no regularization (no dropout, no batch normalization, no weight decay). 
# On clean, well-separated datasets like Iris it performs well — but on noisy datasets like Wine Quality it can collapse entirely, 
# producing 0% test accuracy despite high training accuracy. This is expected behavior and demonstrates why regularization matters. 
# Compare Config D to Config E (heavy regularization) in the winequality or auto_mpg sample dataset to see the effect.

# --- Config A: Conservative baseline ---
config_a = dict(
    name           = 'A: Baseline (Adam+ReLU)',       # type: string  | descriptive name for this config
    layer_sizes    = [n_features, 32, 32, n_classes], # type: list    | [input, hidden..., output]
    actfun         = 'relu',                          # type: string  | relu, tanh, sigmoid, leaky_relu, gelu, elu, selu, silu
    leaky_slope    = 0.01,                            # type: float   | only used for leaky_relu
    learning_rate  = 0.001,                           # type: float   | Adam: 0.001 | SGD: 0.01-0.1
    num_epochs     = 500,                             # type: int     | 100-500 for small datasets, 10-50 for large datasets
    batch_size     = 32,                              # type: int     | powers of 2: 16, 32, 64, 128
    optimizer      = 'adam',                          # type: string  | adam, adamw, sgd, rmsprop, adagrad
    momentum       = 0.9,                             # type: float   | only used for sgd/rmsprop
    loss_function  = 'mse',                    # type: string  | binary: bce_logits | multiclass: cross_entropy | regression: mse, l1, huber
    weight_decay   = 1e-4,                            # type: float   | L2 (0 = off)
    l1_lambda      = 0.0,                             # type: float   | L1 (0 = off)
    dropout_rate   = 0.2,                             # type: float   | 0.0-0.5
    use_batchnorm  = True,                            # type: bool    | True = batch normalization after each hidden layer 
    dev_size       = 0.15,                            # type: float   | dev set for per-epoch monitoring
    test_size      = 0.15,                            # type: float   | held-out test set, touched ONCE
    use_kfold      = False,                           # type: bool    | True = k-fold CV instead of single split
    n_folds        = 5,                               # type: int     | number of folds (only if use_kfold=True)
)

# --- Config B: Deeper network ---
config_b = dict(
    name           = 'B: Deep (4 hidden)',
    layer_sizes    = [n_features, 32, 64, 64, 32, n_classes],
    actfun         = 'relu',
    leaky_slope    = 0.01,
    learning_rate  = 0.001,
    num_epochs     = 500,
    batch_size     = 32,
    optimizer      = 'adam',
    momentum       = 0.9,
    loss_function  = 'mse',
    weight_decay   = 1e-4,
    l1_lambda      = 0.0,
    dropout_rate   = 0.2,
    use_batchnorm  = True,
    dev_size       = 0.15,
    test_size      = 0.15,
    use_kfold      = False,
    n_folds        = 5,
)

# --- Config C: SGD with momentum ---
config_c = dict(
    name           = 'C: SGD+Momentum',
    layer_sizes    = [n_features, 32, 32, n_classes],
    actfun         = 'relu',
    leaky_slope    = 0.01,
    learning_rate  = 0.01,
    num_epochs     = 500,
    batch_size     = 32,
    optimizer      = 'sgd',
    momentum       = 0.9,
    loss_function  = 'mse',
    weight_decay   = 1e-4,
    l1_lambda      = 0.0,
    dropout_rate   = 0.2,
    use_batchnorm  = True,
    dev_size       = 0.15,
    test_size      = 0.15,
    use_kfold      = False,
    n_folds        = 5,
)

# --- Config D: Tanh, no regularization ---
config_d = dict(
    name           = 'D: Tanh, no regularization',
    layer_sizes    = [n_features, 32, 32, n_classes],
    actfun         = 'tanh',
    leaky_slope    = 0.01,
    learning_rate  = 0.001,
    num_epochs     = 500,
    batch_size     = 32,
    optimizer      = 'adam',
    momentum       = 0.9,
    loss_function  = 'mse',
    weight_decay   = 0.0,
    l1_lambda      = 0.0,
    dropout_rate   = 0.0,
    use_batchnorm  = False,
    dev_size       = 0.15,
    test_size      = 0.15,
    use_kfold      = False,
    n_folds        = 5,
)

# --- Config E: Heavy regularization ---
config_e = dict(
    name           = 'E: Heavy regularization',
    layer_sizes    = [n_features, 64, 64, 64, n_classes],
    actfun         = 'relu',
    leaky_slope    = 0.01,
    learning_rate  = 0.001,
    num_epochs     = 500,
    batch_size     = 32,
    optimizer      = 'adamw',
    momentum       = 0.9,
    loss_function  = 'mse',
    weight_decay   = 1e-2,
    l1_lambda      = 1e-4,
    dropout_rate   = 0.4,
    use_batchnorm  = True,
    dev_size       = 0.15,
    test_size      = 0.15,
    use_kfold      = False,
    n_folds        = 5,
)

# ============================================================
# SAMPLE CONFIG MASTER LIST - Do not edit
# ============================================================

config_master = [
    config_a,
    config_b,
    config_c,
    config_d,
    config_e,
    # config_f,
]

### Section 2b - Add Your Own Configurations (EDIT OPTIONAL)

In [ ]:
# (Feel free to create your own configs or copy one from above)
# ============================================================
# CONFIGURATION TEMPLATES — Edit these freely! 
# ============================================================
# Use the template below to create your own configurations by editing the parameters. You can copy-paste the template as many times as you need to create multiple configs.
# For convenience the template includes all parameters with comments on what they do as well as recommended values.

# IMPORTANT: Make sure to include all parameters, even if you set some to 0 or False.
# IMPORTANT: Don't forget to add your new config to the config_master list at the end of this cell!


# For reference, here a description of all parameters:
# -------------------------------------------------------------------------------------------------------------------
#     name           = 'your config name',              # type: string  | Name for display in results table and plots
#     layer_sizes    = [n_features, ... , n_classes],   # type: list    | [input, hidden..., output]
#     actfun         = ' ... ',                         # type: string  | relu, tanh, sigmoid, leaky_relu, gelu, elu, selu, silu
#     leaky_slope    = ... ,                            # type: float   | only used for leaky_relu
#     learning_rate  = ... ,                            # type: float   | Adam: 0.001 | SGD: 0.01-0.1
#     num_epochs     = ... ,                            # type: int     | number of training epochs
#     batch_size     = ... ,                            # type: int     | best practice is powers of 2: 16, 32, 64, 128 (other values also fine, especially for small datasets)
#     optimizer      = ' ... ',                         # type: string  | adam, adamw, sgd, rmsprop, adagrad
#     momentum       = ... ,                            # type: float   | only used for sgd/rmsprop
#     loss_function  = ' ... ',                         # type: string  | binary: bce_logits | multiclass: cross_entropy | regression: mse, l1, huber
#     weight_decay   = ... ,                            # type: float   | L2 (0 = off), start with 1e-4 and adjust based on results (try 0, 1e-5, 1e-4, 1e-3)
#     l1_lambda      = ... ,                            # type: float   | L1 (0 = off), start with 0 and adjust based on results (try 0, 1e-5, 1e-4, 1e-3)
#     dropout_rate   = ... ,                            # type: float   | 0.0-0.5
#     use_batchnorm  = True / False,                    # type: bool    | Whether to use batch normalization after each hidden layer (not recommended for small datasets) 
#     dev_size       = ... ,                            # type: int     | dev set for per-epoch monitoring
#     test_size      = ... ,                            # type: int     | held-out test set, touched ONCE
#     use_kfold      = True / False,                    # type: bool    | True = k-fold CV instead of single split
#     n_folds        = ... ,                            # type: int     | number of folds (only if use_kfold=True)


# --- STARTER TEMPLATE to uncomment, duplicate and edit ---

# config_a = dict(
#     name           = 'your config name',
#     layer_sizes    = [n_features, ... , n_classes],
#     actfun         = ' ... ',
#     leaky_slope    = ... ,
#     learning_rate  = ... ,
#     num_epochs     = ... ,
#     batch_size     = ... ,
#     optimizer      = ' ... ',
#     momentum       = ... ,
#     loss_function  = ' ... ',
#     weight_decay   = ... ,
#     l1_lambda      = ... ,
#     dropout_rate   = ... ,
#     use_batchnorm  = True / False,
#     dev_size       = ... ,
#     test_size      = ... ,
#     use_kfold      = True / False,
#     n_folds        = ... ,
# )


# ============================================================================================
# CONFIG MASTER — Add/remove configs to compare (Do not forget to add all your configs here!)
# ============================================================================================

# config_master = [
#     config_a,
#     ...
# ]

## Section 2c — Verify configs (Do not edit)

In [9]:
# (Do not edit this cell)
# ===================================
# VERIFY FINAL CONFIGURATION SETTINGS
# ===================================

# For sample datasets, ensure loss function is appropriate
fix = False  # set to True to enable print statements for loss function adjustments
if name == 'Iris' and n_classes == 3:
    for i,cfg in enumerate(config_master): 
        if cfg['loss_function'] != 'cross_entropy': 
            cfg['loss_function'] = 'cross_entropy'
            config_master[i] = cfg
            fix = True
            fixtype = 'cross_entropy'
elif name == 'WineQuality-Red' and n_classes == 1:
    for i,cfg in enumerate(config_master): 
        if cfg['loss_function'] != 'bce_logits': 
            cfg['loss_function'] = 'bce_logits'
            config_master[i] = cfg
            fix = True
            fixtype = 'bce_logits'
elif name == 'Auto-mpg' and n_classes == 1:
    for i,cfg in enumerate(config_master): 
        if cfg['loss_function'] not in ['mse', 'l1', 'huber']: 
            cfg['loss_function'] = 'mse'
            config_master[i] = cfg
            fix = True
            fixtype = 'mse'

if fix:
    print(f'⚠️ Adjusted loss functions for sample dataset {name} to match {losstype} requirements (Fix: Set loss function to {fixtype}). \nPlease review the updated configurations below.\n')

spacer = max(len(cfg['name']) for cfg in config_master) + 2

print(f'Begin training {losstype} {name} dataset with {len(config_master)} configurations:\n')
for cfg in config_master:
    mode = f'k-fold (k={cfg["n_folds"]})' if cfg['use_kfold'] else f'single split (dev={cfg["dev_size"]}, test={cfg["test_size"]})'
    print(f'  {cfg["name"]:<{spacer}} | {mode} | loss: {cfg["loss_function"]}, act: {cfg["actfun"]}, opt: {cfg["optimizer"]}')

Begin training regression Auto-mpg dataset with 5 configurations:

  A: Baseline (Adam+ReLU)      | single split (dev=0.15, test=0.15) | loss: mse, act: relu, opt: adam
  B: Deep (4 hidden)           | single split (dev=0.15, test=0.15) | loss: mse, act: relu, opt: adam
  C: SGD+Momentum              | single split (dev=0.15, test=0.15) | loss: mse, act: relu, opt: sgd
  D: Tanh, no regularization   | single split (dev=0.15, test=0.15) | loss: mse, act: tanh, opt: adam
  E: Heavy regularization      | single split (dev=0.15, test=0.15) | loss: mse, act: relu, opt: adamw


---
## Section 3 — Model Engine (do not edit)

In [10]:
# (Do not edit this cell)
# ============================================================
# BUILDING THE MODEL ENGINE
# ============================================================

class FlexibleANN(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        sizes = cfg['layer_sizes']
        self.cfg = cfg
        self.layers = nn.ModuleList()
        self.bnorms = nn.ModuleList()
        for i in range(len(sizes) - 1):
            self.layers.append(nn.Linear(sizes[i], sizes[i+1]))
            if cfg['use_batchnorm'] and i < len(sizes) - 2:
                self.bnorms.append(nn.BatchNorm1d(sizes[i+1]))
            else:
                self.bnorms.append(None)
        self.dropout = nn.Dropout(cfg['dropout_rate']) if cfg['dropout_rate'] > 0 else None

    # Define the activation function based on the config
    def _activate(self, x):
        name = self.cfg['actfun']
        fns = {
            'relu': lambda x: F.relu(x), 'leaky_relu': lambda x: F.leaky_relu(x, negative_slope=self.cfg['leaky_slope']),
            'gelu': lambda x: F.gelu(x), 'elu': lambda x: F.elu(x), 'selu': lambda x: F.selu(x),
            'tanh': lambda x: torch.tanh(x), 'sigmoid': lambda x: torch.sigmoid(x), 'silu': lambda x: F.silu(x),
        }
        if name not in fns:
            raise ValueError(f'Unknown activation: {name}. Options: {list(fns.keys())}')
        return fns[name](x)

    # Define the forward pass
    def forward(self, x):
        for i in range(len(self.layers) - 1):
            x = self.layers[i](x)
            if self.bnorms[i] is not None:
                x = self.bnorms[i](x)
            x = self._activate(x)
            if self.dropout is not None:
                x = self.dropout(x)
        return self.layers[-1](x)

# Build loss function based on config
def _build_lossfun(cfg):
    opts = {
        'bce_logits': nn.BCEWithLogitsLoss(), 'bce': nn.BCELoss(),
        'cross_entropy': nn.CrossEntropyLoss(), 'nll': nn.NLLLoss(),
        'multi_label': nn.MultiLabelSoftMarginLoss(),
        'mse': nn.MSELoss(), 'l1': nn.L1Loss(), 'huber': nn.SmoothL1Loss(),
    }
    if cfg['loss_function'] not in opts:
        raise ValueError(f'Unknown loss: {cfg["loss_function"]}. Options: {list(opts.keys())}')
    return opts[cfg['loss_function']]

# Build optimizer based on config
def _build_optimizer(model, cfg):
    p, lr, wd, mom = model.parameters(), cfg['learning_rate'], cfg['weight_decay'], cfg['momentum']
    opts = {
        'sgd': lambda: torch.optim.SGD(p, lr=lr, weight_decay=wd, momentum=mom),
        'adam': lambda: torch.optim.Adam(p, lr=lr, weight_decay=wd),
        'adamw': lambda: torch.optim.AdamW(p, lr=lr, weight_decay=wd),
        'rmsprop': lambda: torch.optim.RMSprop(p, lr=lr, weight_decay=wd, momentum=mom),
        'adagrad': lambda: torch.optim.Adagrad(p, lr=lr, weight_decay=wd),
    }
    if cfg['optimizer'] not in opts:
        raise ValueError(f'Unknown optimizer: {cfg["optimizer"]}. Options: {list(opts.keys())}')
    return opts[cfg['optimizer']]()

# Compute accuracy (classification) or R² (regression) scaled to 0-100
def _compute_acc(yHat, y, cfg):
    """Compute accuracy (classification) or R² scaled to 0-100 (regression) for a batch."""
    is_regression = cfg['loss_function'] in ('mse', 'l1', 'huber')
    is_binary = cfg['loss_function'] in ('bce_logits', 'bce')

    if is_regression:
        # R² (coefficient of determination) scaled to 0-100
        # R² = 1 - SS_res / SS_tot
        yHat_flat = yHat.squeeze()
        y_flat = y.squeeze()
        ss_res = torch.sum((y_flat - yHat_flat) ** 2)
        ss_tot = torch.sum((y_flat - torch.mean(y_flat)) ** 2)
        r2 = 1 - ss_res / (ss_tot + 1e-8)  # small epsilon to avoid division by zero
        # return 100 * r2.item()           # original R² (can be high negative accuracy number for small samplesizes and specific target distributions)
        return max(0, 100 * r2.item())     # scaled R² with floor at 0 (negatice values clipped) to avoid misleading negative accuracy values in plots
    elif is_binary:
        preds = (yHat > 0).float()
    else:
        preds = torch.argmax(yHat, axis=1)
        y = y.squeeze()
    return 100 * torch.mean((preds == y).float()).item()


def _train_one_split(model, lossfun, optimizer, train_loader, dev_loader, cfg):
    """Train on one split. Dev set used ONLY for monitoring."""
    num_epochs = cfg['num_epochs']
    losses, trainAcc, devAcc = [], [], []
    nweights = sum(p.numel() for n, p in model.named_parameters() if 'weight' in n)

    for epochi in range(num_epochs):
        model.train()
        bAcc, bLoss = [], []
        for X, y in train_loader:
            yHat = model(X)
            loss = lossfun(yHat, y)
            if cfg['l1_lambda'] > 0:
                l1 = sum(torch.sum(torch.abs(p)) for n, p in model.named_parameters() if 'weight' in n)
                loss = loss + cfg['l1_lambda'] * l1 / nweights
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            bLoss.append(loss.item())
            bAcc.append(_compute_acc(yHat, y, cfg))

        trainAcc.append(np.mean(bAcc))
        losses.append(np.mean(bLoss))

        model.eval()
        with torch.no_grad():
            X, y = next(iter(dev_loader))
            yHat = model(X)
            devAcc.append(_compute_acc(yHat, y, cfg))

    return trainAcc, devAcc, losses


def _evaluate_on_test(model, test_loader, cfg):
    """Final evaluation on held-out test set. Called ONCE per config."""
    model.eval()
    with torch.no_grad():
        X, y = next(iter(test_loader))
        yHat = model(X)
        return _compute_acc(yHat, y, cfg)


print('Model engine loaded.')

Model engine loaded.


---
## Section 4 — Run All Configurations (Do not edit)

In [11]:
# (Do not edit this cell)
# ============================================================
# TRAIN BY RUNNING ALL CONFIGS
# ============================================================

results = {}

for cfg in config_master:
    cfg_name = cfg['name']
    print(f'\n{"="*60}')
    print(f'Training: {cfg_name}')
    print(f'  Arch: {cfg["layer_sizes"]}  |  {cfg["actfun"]}  |  {cfg["optimizer"]} lr={cfg["learning_rate"]}')

    t0 = time.time()

    if cfg['use_kfold']:
        print(f'  Mode: {cfg["n_folds"]}-fold cross-validation')
        kf = KFold(n_splits=cfg['n_folds'], shuffle=True, random_state=42)
        fold_trainAcc, fold_devAcc, fold_losses = [], [], []
        fold_final_test = []

        for fold_i, (train_idx, val_idx) in enumerate(kf.split(T_features)):
            print(f'  Fold {fold_i+1}/{cfg["n_folds"]}...', end=' ')
            train_loader = DataLoader(
                TensorDataset(T_features[train_idx], T_labels[train_idx]),
                batch_size=cfg['batch_size'], shuffle=True, drop_last=True)
            val_loader = DataLoader(
                TensorDataset(T_features[val_idx], T_labels[val_idx]),
                batch_size=len(T_features[val_idx]))

            model = FlexibleANN(cfg)
            lossfun = _build_lossfun(cfg)
            optimizer = _build_optimizer(model, cfg)
            trainAcc, devAcc, losses_fold = _train_one_split(model, lossfun, optimizer, train_loader, val_loader, cfg)

            fold_trainAcc.append(trainAcc)
            fold_devAcc.append(devAcc)
            fold_losses.append(losses_fold)
            final_val = _evaluate_on_test(model, val_loader, cfg)
            fold_final_test.append(final_val)
            print(f'val acc: {final_val:.1f}%')

        avg_trainAcc = np.mean(fold_trainAcc, axis=0).tolist()
        avg_devAcc   = np.mean(fold_devAcc, axis=0).tolist()
        avg_losses   = np.mean(fold_losses, axis=0).tolist()
        elapsed = round(time.time() - t0, 2)
        nparams = sum(p.numel() for p in FlexibleANN(cfg).parameters() if p.requires_grad)

        results[cfg_name] = {
            'cfg': cfg, 'trainAcc': avg_trainAcc, 'devAcc': avg_devAcc, 'losses': avg_losses,
            'n_params': nparams, 'train_time': elapsed,
            'best_dev_acc': round(max(avg_devAcc), 2),
            'final_dev_acc': round(avg_devAcc[-1], 2),
            'final_train_acc': round(avg_trainAcc[-1], 2),
            'overfit_gap': round(avg_trainAcc[-1] - avg_devAcc[-1], 2),
            'final_test_acc': round(np.mean(fold_final_test), 2),
            'eval_mode': f'{cfg["n_folds"]}-fold CV',
        }
        print(f'  Mean val acc: {np.mean(fold_final_test):.1f}% +/- {np.std(fold_final_test):.1f}%  |  time: {elapsed}s')

    else:
        print(f'  Mode: 3-way split (train={1-cfg["dev_size"]-cfg["test_size"]:.0%} / dev={cfg["dev_size"]:.0%} / test={cfg["test_size"]:.0%})')
        traindev_data, test_data_split, traindev_labels, test_labels_split = train_test_split(
            T_features, T_labels, test_size=cfg['test_size'], random_state=42)
        dev_fraction = cfg['dev_size'] / (1 - cfg['test_size'])
        train_data_split, dev_data_split, train_labels_split, dev_labels_split = train_test_split(
            traindev_data, traindev_labels, test_size=dev_fraction, random_state=42)

        train_loader = DataLoader(TensorDataset(train_data_split, train_labels_split),
            batch_size=cfg['batch_size'], shuffle=True, drop_last=True)
        dev_loader = DataLoader(TensorDataset(dev_data_split, dev_labels_split), batch_size=len(dev_data_split))
        test_loader = DataLoader(TensorDataset(test_data_split, test_labels_split), batch_size=len(test_data_split))

        model = FlexibleANN(cfg)
        lossfun = _build_lossfun(cfg)
        optimizer = _build_optimizer(model, cfg)
        nparams = sum(p.numel() for p in model.parameters() if p.requires_grad)
        trainAcc, devAcc, losses_run = _train_one_split(model, lossfun, optimizer, train_loader, dev_loader, cfg)
        final_test = _evaluate_on_test(model, test_loader, cfg)
        elapsed = round(time.time() - t0, 2)

        results[cfg_name] = {
            'cfg': cfg, 'trainAcc': trainAcc, 'devAcc': devAcc, 'losses': losses_run,
            'n_params': nparams, 'train_time': elapsed,
            'best_dev_acc': round(max(devAcc), 2),
            'final_dev_acc': round(devAcc[-1], 2),
            'final_train_acc': round(trainAcc[-1], 2),
            'overfit_gap': round(trainAcc[-1] - devAcc[-1], 2),
            'final_test_acc': round(final_test, 2),
            'eval_mode': '3-way split',
        }
        print(f'  Samples: train={len(train_data_split)}, dev={len(dev_data_split)}, test={len(test_data_split)}')
        print(f'  Dev acc:  {devAcc[-1]:.1f}% (best: {max(devAcc):.1f}%)')
        print(f'  Test acc: {final_test:.1f}%')
        print(f'  Time: {elapsed}s  |  Params: {nparams}')

print(f'\n{"="*60}')
print(f'All {len(results)} configurations trained successfully.')


Training: A: Baseline (Adam+ReLU)
  Arch: [7, 32, 32, 1]  |  relu  |  adam lr=0.001
  Mode: 3-way split (train=70% / dev=15% / test=15%)
  Samples: train=274, dev=59, test=59
  Dev acc:  78.0% (best: 84.6%)
  Test acc: 86.0%
  Time: 3.33s  |  Params: 1473

Training: B: Deep (4 hidden)
  Arch: [7, 32, 64, 64, 32, 1]  |  relu  |  adam lr=0.001
  Mode: 3-way split (train=70% / dev=15% / test=15%)
  Samples: train=274, dev=59, test=59
  Dev acc:  84.0% (best: 85.9%)
  Test acc: 90.4%
  Time: 5.01s  |  Params: 9025

Training: C: SGD+Momentum
  Arch: [7, 32, 32, 1]  |  relu  |  sgd lr=0.01
  Mode: 3-way split (train=70% / dev=15% / test=15%)
  Samples: train=274, dev=59, test=59
  Dev acc:  83.7% (best: 87.7%)
  Test acc: 89.7%
  Time: 2.75s  |  Params: 1473

Training: D: Tanh, no regularization
  Arch: [7, 32, 32, 1]  |  tanh  |  adam lr=0.001
  Mode: 3-way split (train=70% / dev=15% / test=15%)
  Samples: train=274, dev=59, test=59
  Dev acc:  85.3% (best: 85.4%)
  Test acc: 88.0%
  Time:

---
## Section 5 — Interactive Comparison Plots (Do not edit)

In [12]:
# (Do not edit this cell)
# ============================================================
# INTERACTIVE COMPARISON
# ============================================================

def smooth(x, k):
    if k <= 1: return x
    return np.convolve(x, np.ones(k)/k, mode='same')

names = list(results.keys())
cmap = plt.cm.tab10
color_map = {n: cmap(i) for i, n in enumerate(names)}

info_parts = [f'<b>{name}</b>', losstype.capitalize(), f'{n_samples} samples', f'{n_features} features']
if losstype in ('binary', 'multiclass'):
    info_parts.append(f'{len(torch.unique(T_labels))} classes')

info_widget = widgets.HTML(
    value=f'<div style="color:black; font-size:16px; text-align:center;">{"  ·  ".join(info_parts)}</div>',
    layout=widgets.Layout(width='600px'))
config_selector = widgets.SelectMultiple(
    options=['ALL'] + names, value=['ALL'], description='Configs:',
    rows=min(len(names)+1, 10), layout=widgets.Layout(width='300px'))
acc_selector = widgets.SelectMultiple(
    options=['train', 'dev'], value=['train', 'dev'], description='Curves:',
    rows=2, layout=widgets.Layout(width='300px'))
smooth_slider = widgets.IntSlider(
    value=1, min=1, max=50, step=1, description='Smoothing:',
    layout=widgets.Layout(width='300px'))
view_selector = widgets.SelectMultiple(
    options=['Losses', 'Accuracy by Epoch', 'Final Accuracy Bar', 'Training Time', 'Time vs Accuracy'],
    value=['Losses', 'Accuracy by Epoch', 'Time vs Accuracy'], description='Views:',
    rows=5, layout=widgets.Layout(width='300px'))


output = widgets.Output()

def update_plot(*args):
    with output:
        clear_output(wait=True)
        sel = list(config_selector.value)
        show = names if 'ALL' in sel else sel
        show_acc = list(acc_selector.value)
        k = smooth_slider.value
        views = list(view_selector.value)
        if not views: return

        fig, axes = plt.subplots(1, len(views), figsize=(7*len(views), 5))
        if len(views) == 1: axes = [axes]
        vi = 0

        if 'Losses' in views:
            ax = axes[vi]; vi += 1
            for n in show:
                ax.plot(smooth(results[n]['losses'], k), color=color_map[n], lw=2, label=n)
            ax.set_title('Loss by Epoch'); ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
            ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

        if 'Accuracy by Epoch' in views:
            ax = axes[vi]; vi += 1
            for n in show:
                c = color_map[n]
                if 'train' in show_acc:
                    ax.plot(smooth(results[n]['trainAcc'], k), color=c, lw=2, ls='-', label=f'{n} (train)')
                if 'dev' in show_acc:
                    ax.plot(smooth(results[n]['devAcc'], k), color=c, lw=2, ls='--', alpha=0.55, label=f'{n} (dev)')
            ax.set_title('Accuracy by Epoch (Train=solid, Dev=dashed)'); ax.set_xlabel('Epoch'); ax.set_ylabel('Accuracy (%)')
            ax.legend(fontsize=7); ax.grid(True, alpha=0.3)

        if 'Final Accuracy Bar' in views:
            ax = axes[vi]; vi += 1
            x_pos = np.arange(len(show)); w = 0.25
            ax.bar(x_pos - w, [results[n]['final_train_acc'] for n in show], w, label='Train',
                   color=[(*color_map[n][:3], 1.0) for n in show])
            ax.bar(x_pos, [results[n]['final_dev_acc'] for n in show], w, label='Dev',
                   color=[(*color_map[n][:3], 0.6) for n in show])
            ax.bar(x_pos + w, [results[n]['final_test_acc'] for n in show], w, label='Test',
                   color=[(*color_map[n][:3], 0.3) for n in show])
            ax.set_xticks(x_pos)
            ax.set_xticklabels([n.split(':')[0] for n in show], rotation=45, ha='right')
            ax.set_ylabel('Accuracy (%)'); ax.set_title('Final Accuracy (Train / Dev / Test)'); ax.legend()

        if 'Training Time' in views:
            ax = axes[vi]; vi += 1
            times = [results[n]['train_time'] for n in show]
            bars = ax.barh(range(len(show)), times, color=[color_map[n] for n in show], height=0.5)
            ax.set_yticks(range(len(show))); ax.set_yticklabels(show, fontsize=8)
            ax.set_xlabel('Time (s)'); ax.set_title('Training Time')
            ax.set_xlim(0, max(times) * 1.25)
            if len(show) == 1:
                ax.set_ylim(-0.5, 1.5)
            for bar, t in zip(bars, times):
                ax.text(bar.get_width() + max(times)*0.02, bar.get_y() + bar.get_height()/2,
                       f'{t:.1f}s', va='center', fontsize=9)

        if 'Time vs Accuracy' in views:
            ax = axes[vi]; vi += 1
            for n in show:
                r = results[n]
                ax.scatter(r['train_time'], r['final_test_acc'], s=120, color=color_map[n],
                          edgecolors='black', lw=0.5, zorder=5)
                ax.annotate(n.split(':')[0], (r['train_time'], r['final_test_acc']),
                           textcoords='offset points', xytext=(8, 4), fontsize=9)
            ax.set_xlabel('Training Time (s)'); ax.set_ylabel('Final Test Accuracy (%)')
            ax.set_title('Time vs Test Accuracy'); ax.grid(True, alpha=0.3)

        plt.tight_layout(); plt.show()

for w in [config_selector, acc_selector, smooth_slider, view_selector]:
    w.observe(update_plot, names='value')

display(widgets.VBox([info_widget, widgets.HBox([config_selector, view_selector]), widgets.HBox([smooth_slider, acc_selector])]), output)

update_plot()

Output()

---
## Section 6 — Summary (Do not edit)

In [ ]:
# (Do not edit this cell)
# ============================================================
# SUMMARY TABLE WITH DIAGNOSTIC LABELS
# ============================================================

rows = []
for cfg_name, r in results.items():
    rows.append({
        'Config': cfg_name,
        'Eval Mode': r['eval_mode'],
        'Best Dev Acc (%)': r['best_dev_acc'],
        'Final Dev Acc (%)': r['final_dev_acc'],
        'Final Test Acc (%)': r['final_test_acc'],
        'Final Train Acc (%)': r['final_train_acc'],
        'Overfit Gap (%)': r['overfit_gap'],
        'Parameters': r['n_params'],
        'Time (s)': r['train_time'],
    })

summary_df = pd.DataFrame(rows).sort_values('Final Test Acc (%)', ascending=False).reset_index(drop=True)

# --- Add diagnostic labels (speed) ---
labels_col = []
times_sorted = sorted(summary_df['Time (s)'].unique())
speed_map = {}
if len(times_sorted) >= 1: speed_map[times_sorted[0]] = 'Fastest'
if len(times_sorted) >= 2: speed_map[times_sorted[1]] = '2nd Fastest'
if len(times_sorted) >= 3: speed_map[times_sorted[2]] = '3rd Fastest'

best_idx = summary_df['Final Test Acc (%)'].idxmax()

# --- Add diagnostic labels (accuracy and fitting) ---
for idx, row in summary_df.iterrows():
    tags = []
    if row['Overfit Gap (%)'] > 10: tags.append('Overfitting')
    elif row['Overfit Gap (%)'] > 5: tags.append('Mild Overfitting')
    if abs(row['Overfit Gap (%)']) < 3 and row['Final Test Acc (%)'] > 60: tags.append('Good Generalization')
    if losstype != 'regression' and row['Final Train Acc (%)'] < 60: tags.append('Low Train Accuracy')
    if losstype == 'regression' and row['Final Train Acc (%)'] < 0: tags.append('Low Train Accuracy (negative R²)')
    if row['Time (s)'] in speed_map: tags.append(speed_map[row['Time (s)']])
    if idx == best_idx: tags.append('Best Test Accuracy')
    labels_col.append(' | '.join(tags) if tags else '-')

summary_df['Diagnostic_labels'] = labels_col

spacer = 47 + len(name)  # adjust spacer based on dataset name length for better formatting
print('='*spacer)
print(' CONFIGURATION COMPARISON SUMMARY FOR ' + name.upper() + ' DATASET ')
print('='*spacer + '\n')

summary_df

 CONFIGURATION COMPARISON SUMMARY FOR AUTO-MPG DATASET 



,Config,Eval Mode,Best Dev Acc (%),Final Dev Acc (%),Final Test Acc (%),Final Train Acc (%),Overfit Gap (%),Parameters,Time (s),Diagnostic_labels
0,B: Deep (4 hidden),3-way split,85.88,84.01,90.42,74.51,-9.50,9025,5.01,Best Test Accuracy
1,C: SGD+Momentum,3-way split,87.71,83.72,89.74,77.39,-6.33,1473,2.75,2nd Fastest
2,"D: Tanh, no regularization",3-way split,85.38,85.34,87.97,90.32,4.98,1345,1.39,Fastest
3,E: Heavy regularization,3-way split,85.46,81.63,87.71,64.00,-17.63,9281,3.69,-
4,A: Baseline (Adam+ReLU),3-way split,84.55,78.00,86.00,76.62,-1.39,1473,3.33,Good Generalization | 3rd Fastest
